In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 加载环境变量
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")


model = init_chat_model("groq:llama-3.3-70b-versatile", api_key=GROQ_API_KEY)

In [9]:
# 方式 1：消息对象（啰嗦）

messages_obj = [
    SystemMessage(content="你是 Python 导师"),
    HumanMessage(content="什么是列表？")
]
# response = model.invoke(messages_obj)
# print(response.content[:100])

print("-------------------")

# 方式 2：字典格式（推荐，简洁）

messages_dict = [
    {"role": "system", "content": "你是 Python 导师"},
    {"role": "user", "content": "什么是列表？"}
]
response = model.invoke(messages_dict)
# print(messages_dict)
print(messages_dict[0])
print(messages_dict[0].get("role"))
print(messages_dict[1].get("role"))
# print(response.content[:100])


-------------------
{'role': 'system', 'content': '你是 Python 导师'}
system
user


In [10]:
# 初始化对话历史
conversation = [
    {"role": "system", "content": "你是一个简洁的助手，回答限制在50字内"}
]

# 第一轮
print("\n【第 1 轮】")
conversation.append({"role": "user", "content": "什么是 Python？"})
print(f"用户: {conversation[-1]['content']}")

r1 = model.invoke(conversation)
print(f"AI: {r1.content}")

# 关键：保存 AI 回复到历史
conversation.append({"role": "assistant", "content": r1.content})

# 第二轮（测试记忆）
print("\n【第 2 轮】")
conversation.append({"role": "user", "content": "它有什么特点？"})
print(f"用户: {conversation[-1]['content']}")

r2 = model.invoke(conversation)
print(f"AI: {r2.content}")

conversation.append({"role": "assistant", "content": r2.content})

# 第三轮（测试上下文）
print("\n【第 3 轮】")
conversation.append({"role": "user", "content": "我第一个问题问的是什么？"})
print(f"用户: {conversation[-1]['content']}")

r3 = model.invoke(conversation)
print(f"AI: {r3.content}")

print(f"\n💡 对话历史共 {len(conversation)} 条消息")
print("   AI 记住了之前的内容，因为每次都传递了完整历史！")


【第 1 轮】
用户: 什么是 Python？
AI: Python 是一种高级编程语言，用于开发、数据分析和机器学习。

【第 2 轮】
用户: 它有什么特点？
AI: 易读、易学、跨平台、灵活、可扩展。

【第 3 轮】
用户: 我第一个问题问的是什么？
AI: 你问的是“什么是Python？”

💡 对话历史共 6 条消息
   AI 记住了之前的内容，因为每次都传递了完整历史！


In [15]:
conversation = [
    {"role":"user","content":"我叫张三"}
]
r1 = model.invoke("我叫张三")
conversation.append(r1)
print(f"用户: 我叫张三")
print(f"AI: {r1.content[:50]}...")

# 第二次（没有传递历史）
r2 = model.invoke(conversation)
print(r2.content)

r3 = model.invoke("我叫什么名字")
print(r3.content)

用户: 我叫张三
AI: 你好，张三。很高兴认识你。今天我可以帮你什么忙吗？...

我担心我不知道你的名字。我们刚刚开始对话，我还没有收到关于你的任何信息。你想告诉我你的名字吗？


In [19]:
def keep_recent_messages(messages, max_pairs=3):
    """
    保留最近的 N 轮对话

    参数:
        messages: 完整消息列表
        max_pairs: 保留的对话轮数

    返回:
        优化后的消息列表
    """
    # 分离 system 消息和对话消息
    system_msgs = [m for m in messages if m.get("role") == "system"]
    conversation_msgs = [m for m in messages if m.get("role") != "system"]
    print("--system_msgs--",system_msgs)
    print("--conversation_msgs--",conversation_msgs)
    # 只保留最近的消息（每轮 = user + assistant）
    max_messages = max_pairs * 2
    recent_msgs = conversation_msgs[-max_messages:]

    print("--recent_msgs--",recent_msgs)

    msg=system_msgs + recent_msgs
    print("msg",msg)
    # 返回：system + 最近对话
    return system_msgs + recent_msgs

# 模拟长对话
long_conversation = [
    {"role": "system", "content": "你是助手"},
    {"role": "user", "content": "第1个问题"},
    {"role": "assistant", "content": "第1个回答"},
    {"role": "user", "content": "第2个问题"},
    {"role": "assistant", "content": "第2个回答"},
    {"role": "user", "content": "第3个问题"},
    {"role": "assistant", "content": "第3个回答"},
    {"role": "user", "content": "第4个问题"},
    {"role": "assistant", "content": "第4个回答"},
    {"role": "user", "content": "第5个问题"},
]

print(f"原始消息数: {len(long_conversation)}")

# 优化：只保留最近 2 轮
optimized = keep_recent_messages(long_conversation, max_pairs=2)
print(f"优化后消息数: {len(optimized)}")
print(f"保留的内容: system + 最近2轮对话")

# 使用优化后的历史
response = model.invoke(optimized)
print(f"\nAI 回复: {response.content[:100]}...")



原始消息数: 10
--system_msgs-- [{'role': 'system', 'content': '你是助手'}]
--conversation_msgs-- [{'role': 'user', 'content': '第1个问题'}, {'role': 'assistant', 'content': '第1个回答'}, {'role': 'user', 'content': '第2个问题'}, {'role': 'assistant', 'content': '第2个回答'}, {'role': 'user', 'content': '第3个问题'}, {'role': 'assistant', 'content': '第3个回答'}, {'role': 'user', 'content': '第4个问题'}, {'role': 'assistant', 'content': '第4个回答'}, {'role': 'user', 'content': '第5个问题'}]
--recent_msgs-- [{'role': 'assistant', 'content': '第3个回答'}, {'role': 'user', 'content': '第4个问题'}, {'role': 'assistant', 'content': '第4个回答'}, {'role': 'user', 'content': '第5个问题'}]
msg [{'role': 'system', 'content': '你是助手'}, {'role': 'assistant', 'content': '第3个回答'}, {'role': 'user', 'content': '第4个问题'}, {'role': 'assistant', 'content': '第4个回答'}, {'role': 'user', 'content': '第5个问题'}]
优化后消息数: 5
保留的内容: system + 最近2轮对话

AI 回复: 似乎我们还没开始对话。你今天能得到什么帮助？...


In [21]:
conversation = [
    {"role": "system", "content": "你是一个友好的助手"}
]

questions = [
    "我叫李明，今年25岁",
    "我喜欢编程",
    "我叫什么名字？",
    "我今年多大？",
    "我喜欢什么？"
]

for i, q in enumerate(questions, 1):
    print(f"\n--- 第 {i} 轮 ---")
    print(f"用户: {q}")

    print("---before---",conversation)
    conversation.append({"role": "user", "content": q})
    print("---after---",conversation)
    response = model.invoke(conversation)

    print(f"AI: {response.content}")
    conversation.append({"role": "assistant", "content": response.content})
    print("---afterAI---",conversation)
print(f"\n💡 总共 {len(conversation)} 条消息")
print("   AI 完美记住了所有信息！")


--- 第 1 轮 ---
用户: 我叫李明，今年25岁
---before--- [{'role': 'system', 'content': '你是一个友好的助手'}]
---after--- [{'role': 'system', 'content': '你是一个友好的助手'}, {'role': 'user', 'content': '我叫李明，今年25岁'}]
AI: 你好，李明。很高兴认识你。25岁是个很棒的年龄，你的生活和事业都刚刚开始。是什么让你来到这里？你想聊什么？工作、爱好还是其他事情？我在这里倾听并提供帮助。
---afterAI--- [{'role': 'system', 'content': '你是一个友好的助手'}, {'role': 'user', 'content': '我叫李明，今年25岁'}, {'role': 'assistant', 'content': '你好，李明。很高兴认识你。25岁是个很棒的年龄，你的生活和事业都刚刚开始。是什么让你来到这里？你想聊什么？工作、爱好还是其他事情？我在这里倾听并提供帮助。'}]

--- 第 2 轮 ---
用户: 我喜欢编程
---before--- [{'role': 'system', 'content': '你是一个友好的助手'}, {'role': 'user', 'content': '我叫李明，今年25岁'}, {'role': 'assistant', 'content': '你好，李明。很高兴认识你。25岁是个很棒的年龄，你的生活和事业都刚刚开始。是什么让你来到这里？你想聊什么？工作、爱好还是其他事情？我在这里倾听并提供帮助。'}]
---after--- [{'role': 'system', 'content': '你是一个友好的助手'}, {'role': 'user', 'content': '我叫李明，今年25岁'}, {'role': 'assistant', 'content': '你好，李明。很高兴认识你。25岁是个很棒的年龄，你的生活和事业都刚刚开始。是什么让你来到这里？你想聊什么？工作、爱好还是其他事情？我在这里倾听并提供帮助。'}, {'role': 'user', 'content': '我喜欢编程'}]
AI: 太棒了，李明！编程是一项很棒的